In [1]:
import pandas as pd
from openpyxl import load_workbook
import os

wb = load_workbook('../data/raw/demographics_report_from_LinkedIn.xlsx', read_only=True)
ws = wb.active
rows = list(ws.iter_rows(values_only=True))

# find where each section starts (row where col B == 'Impressions')
sec_starts = []
for i, row in enumerate(rows):
    if row[1] == 'Impressions':
        sec_starts.append(i)

# skipping DMA -- US only, not relevant
keep_sections = [
    'Company Industry Segment',
    'Location Segment',
    'Job Seniority Segment',
    'Job Title Segment',
    'Job Function Segment',
    'Company Size Segment',
    'Contextual Country/Region Segment',
]

os.makedirs('../data/processed/PowerBI', exist_ok=True)

for idx, start in enumerate(sec_starts):
    end = sec_starts[idx+1] if idx+1 < len(sec_starts) else len(rows)
    sec_name = rows[start][0]
    # print(sec_name)  # used this to debug section names

    if sec_name not in keep_sections:
        continue

    headers = [h for h in rows[start] if h is not None]

    data = []
    for row in rows[start+1:end]:
        if row[0] is not None and row[1] is not None:
            data.append(row[:len(headers)])

    df = pd.DataFrame(data, columns=headers)

    wanted = [df.columns[0], 'Impressions', 'Clicks', 'Conversions',
              'Click Through Rate', 'Conversion Rate']
    df = df[[c for c in wanted if c in df.columns]]
    df = df.rename(columns={df.columns[0]: df.columns[0].replace(' Segment', '').strip()})
    df = df.sort_values('Impressions', ascending=False).reset_index(drop=True)

    fname = sec_name.replace(' ', '_').replace('/', '_') + '.csv'
    df.to_csv(f'../data/processed/PowerBI/{fname}', index=False)
    print(f'{fname} -- {len(df)} rows')
#   print(df.head(3).to_string()) -------> commented out for confidentiality
    print()

Company_Industry_Segment.csv -- 25 rows

Company_Size_Segment.csv -- 9 rows

Contextual_Country_Region_Segment.csv -- 25 rows

Location_Segment.csv -- 25 rows

Job_Seniority_Segment.csv -- 10 rows

Job_Title_Segment.csv -- 25 rows

Job_Function_Segment.csv -- 25 rows

